# Neido Studio · Voces, movimiento y labios (prueba técnica)
1. Comprueba arriba a la derecha que pone **T4**.
2. **Entorno de ejecución → Ejecutar todo**.
3. Tarda entre 1 y 2 horas. No cierres la pestaña ni dejes que el PC se suspenda.
4. Al final se descarga **neido-video.zip**: mándaselo a Claude.


In [ ]:
# Neido Studio · Etapas "voces" + "video" + "labios" en Google Colab (GPU T4 gratuita).
# 0) Voces con Chatterbox Multilingual (MIT) a partir de voces semilla de Kokoro (Apache 2.0), en un entorno
#    aparte (uv, Python 3.11) porque Chatterbox fija versiones antiguas de torch y diffusers.
# 1) Wan 2.2 TI2V-5B (Apache 2.0) anima el fotograma clave de cada plano "video" / "lipsync".
#    Para que quepa en la T4 (16 GB de vídeo, 12 GB de RAM): primero se calculan los textos con el
#    codificador T5 y se libera; luego se carga solo el modelo de vídeo en fp16.
# 2) LatentSync 1.5 (ByteDance, pesos OpenRAIL++) sincroniza los labios de los planos "lipsync" con su voz.
#    Va en un entorno aparte (uv, Python 3.10) porque fija sus propias versiones de librerías.
# Los fotogramas y las voces vienen incrustados en el cuaderno ('eyJwbGFub3MiOiB7InRpdHVsbyI6ICJQcnVlYmEgdFx1MDBlOWNuaWNhIFx1MDBiNyBUaXRhbmljIiwgInJvdHVsbyI6ICJBdGxcdTAwZTFudGljbyBOb3J0ZSBcdTAwYjcgMTQgZGUgYWJyaWwgZGUgMTkxMiIsICJlc3RpbG8iOiAiQ2luZW1hdGljIGZpbG0gc3RpbGwsIHBob3RvcmVhbGlzdGljLCBzaG90IG9uIDM1bW0gd2l0aCBhIGNpbmVtYSBjYW1lcmEsIG5hdHVyYWwgbW90aXZhdGVkIGxpZ2h0aW5nLCByZWFsaXN0aWMgc2tpbiB0ZXh0dXJlIHdpdGggcG9yZXMgYW5kIGltcGVyZmVjdGlvbnMsIHN1YnRsZSBmaWxtIGdyYWluLCBzbGlnaHRseSBtdXRlZCBuYXR1cmFsIGNvbG91cnMsIHNoYWxsb3cgZGVwdGggb2YgZmllbGQsIGF0bW9zcGhlcmljIGhhemUsIGNvbXBvc2VkIGZvciBhIDIuMzk6MSB3aWRlc2NyZWVuIGNyb3Agd2l0aCB0aGUgaW1wb3J0YW50IGFjdGlvbiBpbiB0aGUgY2VudHJhbCBiYW5kIiwgIm5lZ2F0aXZvIjogInRleHQsIHdhdGVybWFyaywgbG9nbywgbmVvbiwgb3ZlcnNhdHVyYXRlZCwgcGxhc3RpYyBza2luLCBjYXJ0b29uLCBpbGx1c3RyYXRpb24sIGRlZm9ybWVkIGhhbmRzLCBleHRyYSBmaW5nZXJzLCBibHVycnkgZmFjZSwgc3R1ZGlvIGJhY2tkcm9wLCBwbGFpbiBncmV5IHdhbGwsIHNlYW1sZXNzIHBhcGVyIGJhY2tncm91bmQsIGRheWxpZ2h0LCBzdW5ueSIsICJwZXJzb25hamVzIjogeyJzYW50aSI6IHsiZGVzY3JpcGNpb24iOiAiYSAzMi15ZWFyLW9sZCBDb2xvbWJpYW4gbWFuLCBsaWdodCBicm93biBza2luLCBzaG9ydCBibGFjayBoYWlyIHdpdGggc2hhdmVkIHNpZGVzLCBzaG9ydCBzdHViYmxlIGJlYXJkLCBicm93biBleWVzLCBzbGltIGJ1aWxkLCB3ZWFyaW5nIGEgcGxhaW4geWVsbG93IHQtc2hpcnQsIGFuIG9wZW4gYmx1ZSBkZW5pbSBqYWNrZXQsIGRhcmsgamVhbnMsIHdoaXRlIHNuZWFrZXJzIGFuZCBhIHBsYWluIGJsYWNrIGJhc2ViYWxsIGNhcCIsICJ2b3oiOiB7ImlkaW9tYSI6ICJlcyIsICJhY2VudG8iOiAiY29sb21iaWFubyBkZSBNZWRlbGxcdTAwZWRuIiwgInRvbm8iOiAibWFzY3VsaW5hLCAzMC0zNSBhXHUwMGYxb3MsIGNcdTAwZTFsaWRhLCByXHUwMGUxcGlkYSwgY29uIGh1bW9yIn19LCAibWFyZ2FyZXQiOiB7ImRlc2NyaXBjaW9uIjogImEgNTUteWVhci1vbGQgQnJpdGlzaCBhcmlzdG9jcmF0aWMgd29tYW4gaW4gMTkxMiwgZmFpciBza2luLCBncmV5IGhhaXIgaW4gYSBoaWdoIGJ1biwgYW5ndWxhciBmYWNlIHdpdGggbWFya2VkIHdyaW5rbGVzLCBncmV5LWJsdWUgZXllcywgd2VhcmluZyBhIGRhcmsgYnVyZ3VuZHkgRWR3YXJkaWFuIGV2ZW5pbmcgZ293biwgYSBsaWdodCBmdXIgc3RvbGUsIGEgcGVhcmwgbmVja2xhY2UgYW5kIGxvbmcgaXZvcnkgZ2xvdmVzIiwgInZveiI6IHsiaWRpb21hIjogImVzIiwgImFjZW50byI6ICJlc3BhXHUwMGYxb2wgZm9ybWFsIG5ldXRybyIsICJ0b25vIjogImZlbWVuaW5hLCA1NSBhXHUwMGYxb3MsIGFndWRhLCBhbHRpdmEsIHBhdXNhZGEifX0sICJuYXJyYWRvciI6IHsidm96IjogeyJpZGlvbWEiOiAiZXMiLCAiYWNlbnRvIjogImVzcGFcdTAwZjFvbCBuZXV0cm8iLCAidG9ubyI6ICJtYXNjdWxpbmEsIGdyYXZlLCBzZXJlbmEsIGFsZ28gbWlzdGVyaW9zYSJ9fX0sICJwbGFub3MiOiBbeyJuIjogMSwgImR1cmFjaW9uIjogOC42LCAidGlwbyI6ICJpbWFnZW5fZmlqYSIsICJtb3ZpbWllbnRvIjogImFjZXJjYW1pZW50byIsICJwZXJzb25hamVzIjogW10sICJwcm9tcHQiOiAiTmlnaHQgc2NlbmUuIFdpZGUgZXN0YWJsaXNoaW5nIHNob3Qgb2YgdGhlIFJNUyBUaXRhbmljIHNhaWxpbmcgYXQgbmlnaHQgb24gYSBwZXJmZWN0bHkgY2FsbSwgbWlycm9yLWxpa2UgTm9ydGggQXRsYW50aWMsIG1vb25sZXNzIHNreSBmdWxsIG9mIHN0YXJzLCB3YXJtIGxpZ2h0cyBnbG93aW5nIGZyb20gdGhlIHBvcnRob2xlcyBhbmQgZGVja3MsIGZhaW50IHNtb2tlIGZyb20gdGhlIGZ1bm5lbHMsIGNvbGQgYXRtb3NwaGVyZSIsICJhdWRpbyI6IFt7InF1aWVuIjogIm5hcnJhZG9yIiwgInRleHRvIjogIkNhdG9yY2UgZGUgYWJyaWwgZGUgbWlsIG5vdmVjaWVudG9zIGRvY2UuIEVzdGEgbm9jaGUsIGVuIGVsIGJhcmNvIG1cdTAwZTFzIGZhbW9zbyBkZWwgbXVuZG8sIGhheSBhbGd1aWVuIHF1ZSBubyBkZWJlclx1MDBlZGEgZXN0YXIgYXF1XHUwMGVkLiJ9XSwgInJvdHVsbyI6IHRydWUsICJhbWJpZW50ZSI6ICJtYXIgZW4gY2FsbWEsIG1vdG9yZXMgbGVqYW5vcyJ9LCB7Im4iOiAyLCAiZHVyYWNpb24iOiAzLjAsICJ0aXBvIjogInZpZGVvIiwgInBlcnNvbmFqZXMiOiBbXSwgInByb21wdCI6ICJOaWdodCBzY2VuZS4gUG9pbnQtb2YtdmlldyBzaG90IGZyb20gaW5zaWRlIGEgZGFyayBuYXJyb3cgc2VydmljZSBjb3JyaWRvciBvZiBhIDE5MTIgb2NlYW4gbGluZXIsIGEgd29vZGVuIGRvb3Igb3BlbmluZyBvbnRvIHRoZSBmaXJzdC1jbGFzcyBwcm9tZW5hZGUgZGVjayBhdCBuaWdodCwgd2FybSBnb2xkZW4gbGlnaHQgZnJvbSB0aGUgZGVjayBsYW1wcyBzcGlsbGluZyBpbiwgYSBmYWludCBjb29sIGN5YW4gZ2xvdyBiZWhpbmQgdGhlIHZpZXdlciwgd29vZGVuIGRlY2sgcGxhbmtzIGFuZCBhIHdoaXRlIHJhaWxpbmcgdmlzaWJsZSB0aHJvdWdoIHRoZSBkb29yLCBzdGFycnkgc2t5IiwgIm1vdmltaWVudG9fdmlkZW8iOiAidGhlIGRvb3Igc2xvd2x5IHN3aW5ncyBvcGVuIGFuZCB0aGUgY2FtZXJhIG1vdmVzIGZvcndhcmQgdGhyb3VnaCBpdCIsICJhdWRpbyI6IFtdLCAiYW1iaWVudGUiOiAicHVlcnRhLCB2aWVudG8ifSwgeyJuIjogMywgImR1cmFjaW9uIjogNC4zLCAidGlwbyI6ICJ2aWRlbyIsICJwZXJzb25hamVzIjogWyJzYW50aSJdLCAidmVydGljYWwiOiB0cnVlLCAicHJvbXB0IjogIk5pZ2h0IHNjZW5lLiBNZWRpdW0gc2hvdCBvZiB7c2FudGl9IHN0ZXBwaW5nIG91dCBvbnRvIHRoZSBmaXJzdC1jbGFzcyBwcm9tZW5hZGUgZGVjayBvZiB0aGUgVGl0YW5pYyBhdCBuaWdodCBpbiAxOTEyLCBsb29raW5nIGFyb3VuZCBjb25mdXNlZCBhbmQgYW1hemVkLCB0aGUgYm93IHRvIHRoZSByaWdodCBvZiB0aGUgZnJhbWUsIHdvb2RlbiBkZWNrLCB3aGl0ZSByYWlsaW5nLCBsaWZlYm9hdHMgaGFuZ2luZyBhYm92ZSwgd2FybSBlbGVjdHJpYyB3YWxsIGxhbXBzLCBicmVhdGggdmlzaWJsZSBpbiB0aGUgY29sZCBhaXIsIHN0YXJyeSBza3kiLCAibW92aW1pZW50b192aWRlbyI6ICJoZSB0YWtlcyBhIHN0ZXAgZm9yd2FyZCBhbmQgdHVybnMgaGlzIGhlYWQgc2xvd2x5LCBsb29raW5nIGFyb3VuZCIsICJhdWRpbyI6IFt7InF1aWVuIjogInNhbnRpIiwgInRpcG8iOiAicGVuc2FtaWVudG8iLCAidGV4dG8iOiAiQXZlIE1hclx1MDBlZGFcdTIwMjYgZXN0byBubyBlcyBlbCBtZXRybyBkZSBNZWRlbGxcdTAwZWRuLiJ9XSwgImFtYmllbnRlIjogInZpZW50bywgbWFyIn0sIHsibiI6IDQsICJkdXJhY2lvbiI6IDIuMCwgInRpcG8iOiAiaW1hZ2VuX2ZpamEiLCAibW92aW1pZW50byI6ICJsZXZlIiwgInBlcnNvbmFqZXMiOiBbInNhbnRpIl0sICJwcm9tcHQiOiAiTmlnaHQgc2NlbmUuIENsb3NlLXVwIGluc2VydCBvZiBhIG1hbidzIGhhbmQgaG9sZGluZyBhIG1vZGVybiBzbWFydHBob25lIHdpdGggYSBibGFjayBjYXNlLCB0aGUgc2NyZWVuIGdsb3dpbmcgY29vbCBibHVlLXdoaXRlIGFuZCBibGFuaywgYWdhaW5zdCB0aGUgZGFyayB3b29kZW4gZGVjayBvZiBhIDE5MTIgb2NlYW4gbGluZXIgYXQgbmlnaHQsIHdhcm0gbGFtcCBsaWdodCBpbiB0aGUgYmFja2dyb3VuZCIsICJzdXBlcnBvc2ljaW9uIjogeyJ0ZXh0byI6ICI0OjA0IiwgInN1YnRleHRvIjogIlNpbiBzZXJ2aWNpbyIsICJ4IjogMC41MTIsICJ5IjogMC40MiwgInRhbWFubyI6IDQ4fSwgImF1ZGlvIjogW10sICJhbWJpZW50ZSI6ICJwaXRpZG8gZGUgZXJyb3Igc3VhdmUifSwgeyJuIjogNSwgImR1cmFjaW9uIjogMy4wLCAidGlwbyI6ICJ2aWRlbyIsICJwZXJzb25hamVzIjogWyJtYXJnYXJldCJdLCAidmVydGljYWwiOiB0cnVlLCAicHJvbXB0IjogIk5pZ2h0IHNjZW5lLiBNZWRpdW0gc2hvdCBvZiB7bWFyZ2FyZXR9IHN0YW5kaW5nIGJ5IHRoZSB3aGl0ZSByYWlsaW5nIG9mIHRoZSBUaXRhbmljJ3MgZmlyc3QtY2xhc3MgcHJvbWVuYWRlIGRlY2sgYXQgbmlnaHQgaW4gMTkxMiwgbG9va2luZyBhdCBzb21lb25lIG9mZi1zY3JlZW4gdG8gdGhlIGxlZnQgd2l0aCBkaXNkYWluIGFuZCBjdXJpb3NpdHksIG9uZSBleWVicm93IHJhaXNlZCwgd2FybSBlbGVjdHJpYyB3YWxsIGxhbXBzLCBzdGFycnkgc2t5LCBjYWxtIGJsYWNrIHNlYSBiZWhpbmQgaGVyLCBicmVhdGggdmlzaWJsZSBpbiB0aGUgY29sZCBhaXIiLCAibW92aW1pZW50b192aWRlbyI6ICJzaGUgc2xvd2x5IGxvb2tzIGhpbSB1cCBhbmQgZG93biIsICJhdWRpbyI6IFtdLCAiYW1iaWVudGUiOiAibVx1MDBmYXNpY2EgZGUgY3VlcmRhIGxlamFuYSJ9LCB7Im4iOiA2LCAiZHVyYWNpb24iOiA1LjYsICJ0aXBvIjogImxpcHN5bmMiLCAicGVyc29uYWplcyI6IFsibWFyZ2FyZXQiXSwgInZlcnRpY2FsIjogdHJ1ZSwgInByb21wdCI6ICJOaWdodCBzY2VuZS4gTWVkaXVtIGNsb3NlLXVwIG9mIHttYXJnYXJldH0gb24gdGhlIFRpdGFuaWMncyBmaXJzdC1jbGFzcyBkZWNrIGF0IG5pZ2h0IGluIDE5MTIsIGZhY2luZyB0aGUgY2FtZXJhIHNsaWdodGx5IHRvIHRoZSBsZWZ0LCBoYXVnaHR5IGV4cHJlc3Npb24sIHdhcm0gbGFtcCBsaWdodCBvbiBoZXIgZmFjZSwgZGFyayBzdGFycnkgYmFja2dyb3VuZCBzb2Z0bHkgYmx1cnJlZCwgZmFjZSBjbGVhcmx5IHZpc2libGUgYW5kIHdlbGwgbGl0IiwgIm1vdmltaWVudG9fdmlkZW8iOiAic2hlIHNwZWFrcyB3aXRoIGEgaGF1Z2h0eSBleHByZXNzaW9uLCBzdWJ0bGUgaGVhZCBtb3ZlbWVudCIsICJhdWRpbyI6IFt7InF1aWVuIjogIm1hcmdhcmV0IiwgInRleHRvIjogIlx1MDBiZlNlIGhhIHBlcmRpZG8sIGNhYmFsbGVybz8gRXN0YSBjdWJpZXJ0YSBlcyBkZSBwcmltZXJhIGNsYXNlLiJ9XSwgImFtYmllbnRlIjogIm1cdTAwZmFzaWNhIGRlIGN1ZXJkYSBsZWphbmEifSwgeyJuIjogNywgImR1cmFjaW9uIjogNi42LCAidGlwbyI6ICJsaXBzeW5jIiwgInBlcnNvbmFqZXMiOiBbInNhbnRpIl0sICJ2ZXJ0aWNhbCI6IHRydWUsICJwcm9tcHQiOiAiTmlnaHQgc2NlbmUuIE1lZGl1bSBjbG9zZS11cCBvZiB7c2FudGl9IG9uIHRoZSBUaXRhbmljJ3MgZmlyc3QtY2xhc3MgZGVjayBhdCBuaWdodCBpbiAxOTEyLCBmYWNpbmcgdGhlIGNhbWVyYSBzbGlnaHRseSB0byB0aGUgcmlnaHQsIG5lcnZvdXMgZnJpZW5kbHkgc21pbGUsIHdhcm0gbGFtcCBsaWdodCBvbiBvbmUgc2lkZSBvZiBoaXMgZmFjZSwgYSBmYWludCBjb29sIGJsdWUgZ2xvdyBmcm9tIGJlbG93IGZyb20gaGlzIHBob25lLCBkYXJrIHN0YXJyeSBiYWNrZ3JvdW5kIHNvZnRseSBibHVycmVkLCBmYWNlIGNsZWFybHkgdmlzaWJsZSBhbmQgd2VsbCBsaXQiLCAibW92aW1pZW50b192aWRlbyI6ICJoZSBzcGVha3Mgd2l0aCBhIG5lcnZvdXMgZnJpZW5kbHkgc21pbGUsIHNtYWxsIGdlc3R1cmVzIiwgImF1ZGlvIjogW3sicXVpZW4iOiAic2FudGkiLCAidGV4dG8iOiAiXHUwMGJmUXVcdTAwZTkgcGFzYSwgcHVlcz8gVHJhbnF1aWxhLCBtaSBzZVx1MDBmMW9yYVx1MjAyNiBPaWdhLCB1bmEgcHJlZ3VudGljYTogXHUwMGJmZXN0ZSBiYXJjbyBjXHUwMGYzbW8gc2UgbGxhbWE/In1dLCAiYW1iaWVudGUiOiAidmllbnRvIn0sIHsibiI6IDgsICJkdXJhY2lvbiI6IDUuMywgInRpcG8iOiAidmlkZW8iLCAicGVyc29uYWplcyI6IFsibWFyZ2FyZXQiXSwgInByb21wdCI6ICJOaWdodCBzY2VuZS4gTWVkaXVtIHNob3QgaW4gcHJvZmlsZSBvZiB7bWFyZ2FyZXR9IGxvb2tpbmcgcHJvdWRseSBvdXQgYXQgdGhlIGNhbG0gYmxhY2sgc2VhIGZyb20gdGhlIFRpdGFuaWMncyByYWlsaW5nIGF0IG5pZ2h0IGluIDE5MTIsIGNoaW4gcmFpc2VkLCB3YXJtIGxhbXAgbGlnaHQgb3V0bGluaW5nIGhlciBwcm9maWxlLCBzdGFycnkgc2t5IiwgIm1vdmltaWVudG9fdmlkZW8iOiAic2hlIGxpZnRzIGhlciBjaGluIHByb3VkbHksIGFsbW9zdCBzdGlsbCIsICJhdWRpbyI6IFt7InF1aWVuIjogIm1hcmdhcmV0IiwgInRleHRvIjogIkVsIFRpdGFuaWMsIHBvciBzdXB1ZXN0by4gRWwgYmFyY28gbVx1MDBlMXMgc2VndXJvIGphbVx1MDBlMXMgY29uc3RydWlkby4ifV0sICJhbWJpZW50ZSI6ICJtYXIifSwgeyJuIjogOSwgImR1cmFjaW9uIjogNC4wLCAidGlwbyI6ICJ2aWRlbyIsICJwZXJzb25hamVzIjogWyJzYW50aSJdLCAidmVydGljYWwiOiB0cnVlLCAicHJvbXB0IjogIk5pZ2h0IHNjZW5lLiBDbG9zZS11cCBvZiB0aGUgZmFjZSBvZiB7c2FudGl9IG9uIHRoZSBUaXRhbmljJ3MgZGVjayBhdCBuaWdodCBpbiAxOTEyLCBoaXMgZXhwcmVzc2lvbiBjb2xsYXBzaW5nIGludG8gc2xvd2x5IGRhd25pbmcgaG9ycm9yLCBleWVzIHdpZGUsIHRoZSBzbWlsZSBkaXNhcHBlYXJpbmcsIHdhcm0gbGFtcCBsaWdodCBmcm9tIHRoZSBzaWRlLCBkYXJrIGJhY2tncm91bmQiLCAibW92aW1pZW50b192aWRlbyI6ICJoaXMgc21pbGUgc2xvd2x5IGZhZGVzIGludG8gaG9ycm9yLCBleWVzIHdpZGVuaW5nIiwgImF1ZGlvIjogW3sicXVpZW4iOiAic2FudGkiLCAidGlwbyI6ICJwZW5zYW1pZW50byIsICJ0ZXh0byI6ICJBeSwgbm8uIE5vLCBubywgbm8sIG5vXHUyMDI2In1dLCAiYW1iaWVudGUiOiAic2lsZW5jaW8iLCAiZmluYWwiOiAibmVncm8ifV19LCAiYXJjaGl2b3MiOiB7InBsYW5vXzAyLmpwZyI6ICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbmVpZG9zdG9yZS9DYW5hbC15b3V0dWJlL2Njci0wMDY0NGU4Yi0xNGI0amkvcGVsaWN1bGFzLWlhL3Byb2R1Y2Npb25lcy9wcnVlYmEtdGVjbmljYS9pbWFnZW5lcy1jb2xhYi12My9wbGFub18wMi5qcGciLCAicGxhbm9fMDMuanBnIjogImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9uZWlkb3N0b3JlL0NhbmFsLXlvdXR1YmUvY2NyLTAwNjQ0ZThiLTE0YjRqaS9wZWxpY3VsYXMtaWEvcHJvZHVjY2lvbmVzL3BydWViYS10ZWNuaWNhL2ltYWdlbmVzLWNvbGFiLXYzL3BsYW5vXzAzLmpwZyIsICJwbGFub18wNS5qcGciOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvaW1hZ2VuZXMtY29sYWItdjMvcGxhbm9fMDUuanBnIiwgInBsYW5vXzA2LmpwZyI6ICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbmVpZG9zdG9yZS9DYW5hbC15b3V0dWJlL2Njci0wMDY0NGU4Yi0xNGI0amkvcGVsaWN1bGFzLWlhL3Byb2R1Y2Npb25lcy9wcnVlYmEtdGVjbmljYS9pbWFnZW5lcy1jb2xhYi12My9wbGFub18wNi5qcGciLCAicGxhbm9fMDcuanBnIjogImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9uZWlkb3N0b3JlL0NhbmFsLXlvdXR1YmUvY2NyLTAwNjQ0ZThiLTE0YjRqaS9wZWxpY3VsYXMtaWEvcHJvZHVjY2lvbmVzL3BydWViYS10ZWNuaWNhL2ltYWdlbmVzLWNvbGFiLXYzL3BsYW5vXzA3LmpwZyIsICJwbGFub18wOC5qcGciOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvaW1hZ2VuZXMtY29sYWItdjMvcGxhbm9fMDguanBnIiwgInBsYW5vXzA5LmpwZyI6ICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbmVpZG9zdG9yZS9DYW5hbC15b3V0dWJlL2Njci0wMDY0NGU4Yi0xNGI0amkvcGVsaWN1bGFzLWlhL3Byb2R1Y2Npb25lcy9wcnVlYmEtdGVjbmljYS9pbWFnZW5lcy1jb2xhYi12My9wbGFub18wOS5qcGciLCAicGxhbm9fMDFfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDFfMS53YXYiLCAicGxhbm9fMDNfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDNfMS53YXYiLCAicGxhbm9fMDZfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDZfMS53YXYiLCAicGxhbm9fMDdfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDdfMS53YXYiLCAicGxhbm9fMDhfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDhfMS53YXYiLCAicGxhbm9fMDlfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDlfMS53YXYifSwgImR1cl92b3oiOiB7fX0='). Salida: /content/entrega
import base64, gc, json, os, subprocess, sys, time, traceback
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TQDM_DISABLE"] = "1"   # menos texto: el navegador no se atasca
T0 = time.time(); LIMITE = 2.6 * 3600          # Colab gratis corta sesiones largas: paramos a tiempo
OUT = "/content/entrega"; IN = "/content/entrada"; os.makedirs(OUT, exist_ok=True); os.makedirs(IN, exist_ok=True)
DATOS = json.loads(base64.b64decode('eyJwbGFub3MiOiB7InRpdHVsbyI6ICJQcnVlYmEgdFx1MDBlOWNuaWNhIFx1MDBiNyBUaXRhbmljIiwgInJvdHVsbyI6ICJBdGxcdTAwZTFudGljbyBOb3J0ZSBcdTAwYjcgMTQgZGUgYWJyaWwgZGUgMTkxMiIsICJlc3RpbG8iOiAiQ2luZW1hdGljIGZpbG0gc3RpbGwsIHBob3RvcmVhbGlzdGljLCBzaG90IG9uIDM1bW0gd2l0aCBhIGNpbmVtYSBjYW1lcmEsIG5hdHVyYWwgbW90aXZhdGVkIGxpZ2h0aW5nLCByZWFsaXN0aWMgc2tpbiB0ZXh0dXJlIHdpdGggcG9yZXMgYW5kIGltcGVyZmVjdGlvbnMsIHN1YnRsZSBmaWxtIGdyYWluLCBzbGlnaHRseSBtdXRlZCBuYXR1cmFsIGNvbG91cnMsIHNoYWxsb3cgZGVwdGggb2YgZmllbGQsIGF0bW9zcGhlcmljIGhhemUsIGNvbXBvc2VkIGZvciBhIDIuMzk6MSB3aWRlc2NyZWVuIGNyb3Agd2l0aCB0aGUgaW1wb3J0YW50IGFjdGlvbiBpbiB0aGUgY2VudHJhbCBiYW5kIiwgIm5lZ2F0aXZvIjogInRleHQsIHdhdGVybWFyaywgbG9nbywgbmVvbiwgb3ZlcnNhdHVyYXRlZCwgcGxhc3RpYyBza2luLCBjYXJ0b29uLCBpbGx1c3RyYXRpb24sIGRlZm9ybWVkIGhhbmRzLCBleHRyYSBmaW5nZXJzLCBibHVycnkgZmFjZSwgc3R1ZGlvIGJhY2tkcm9wLCBwbGFpbiBncmV5IHdhbGwsIHNlYW1sZXNzIHBhcGVyIGJhY2tncm91bmQsIGRheWxpZ2h0LCBzdW5ueSIsICJwZXJzb25hamVzIjogeyJzYW50aSI6IHsiZGVzY3JpcGNpb24iOiAiYSAzMi15ZWFyLW9sZCBDb2xvbWJpYW4gbWFuLCBsaWdodCBicm93biBza2luLCBzaG9ydCBibGFjayBoYWlyIHdpdGggc2hhdmVkIHNpZGVzLCBzaG9ydCBzdHViYmxlIGJlYXJkLCBicm93biBleWVzLCBzbGltIGJ1aWxkLCB3ZWFyaW5nIGEgcGxhaW4geWVsbG93IHQtc2hpcnQsIGFuIG9wZW4gYmx1ZSBkZW5pbSBqYWNrZXQsIGRhcmsgamVhbnMsIHdoaXRlIHNuZWFrZXJzIGFuZCBhIHBsYWluIGJsYWNrIGJhc2ViYWxsIGNhcCIsICJ2b3oiOiB7ImlkaW9tYSI6ICJlcyIsICJhY2VudG8iOiAiY29sb21iaWFubyBkZSBNZWRlbGxcdTAwZWRuIiwgInRvbm8iOiAibWFzY3VsaW5hLCAzMC0zNSBhXHUwMGYxb3MsIGNcdTAwZTFsaWRhLCByXHUwMGUxcGlkYSwgY29uIGh1bW9yIn19LCAibWFyZ2FyZXQiOiB7ImRlc2NyaXBjaW9uIjogImEgNTUteWVhci1vbGQgQnJpdGlzaCBhcmlzdG9jcmF0aWMgd29tYW4gaW4gMTkxMiwgZmFpciBza2luLCBncmV5IGhhaXIgaW4gYSBoaWdoIGJ1biwgYW5ndWxhciBmYWNlIHdpdGggbWFya2VkIHdyaW5rbGVzLCBncmV5LWJsdWUgZXllcywgd2VhcmluZyBhIGRhcmsgYnVyZ3VuZHkgRWR3YXJkaWFuIGV2ZW5pbmcgZ293biwgYSBsaWdodCBmdXIgc3RvbGUsIGEgcGVhcmwgbmVja2xhY2UgYW5kIGxvbmcgaXZvcnkgZ2xvdmVzIiwgInZveiI6IHsiaWRpb21hIjogImVzIiwgImFjZW50byI6ICJlc3BhXHUwMGYxb2wgZm9ybWFsIG5ldXRybyIsICJ0b25vIjogImZlbWVuaW5hLCA1NSBhXHUwMGYxb3MsIGFndWRhLCBhbHRpdmEsIHBhdXNhZGEifX0sICJuYXJyYWRvciI6IHsidm96IjogeyJpZGlvbWEiOiAiZXMiLCAiYWNlbnRvIjogImVzcGFcdTAwZjFvbCBuZXV0cm8iLCAidG9ubyI6ICJtYXNjdWxpbmEsIGdyYXZlLCBzZXJlbmEsIGFsZ28gbWlzdGVyaW9zYSJ9fX0sICJwbGFub3MiOiBbeyJuIjogMSwgImR1cmFjaW9uIjogOC42LCAidGlwbyI6ICJpbWFnZW5fZmlqYSIsICJtb3ZpbWllbnRvIjogImFjZXJjYW1pZW50byIsICJwZXJzb25hamVzIjogW10sICJwcm9tcHQiOiAiTmlnaHQgc2NlbmUuIFdpZGUgZXN0YWJsaXNoaW5nIHNob3Qgb2YgdGhlIFJNUyBUaXRhbmljIHNhaWxpbmcgYXQgbmlnaHQgb24gYSBwZXJmZWN0bHkgY2FsbSwgbWlycm9yLWxpa2UgTm9ydGggQXRsYW50aWMsIG1vb25sZXNzIHNreSBmdWxsIG9mIHN0YXJzLCB3YXJtIGxpZ2h0cyBnbG93aW5nIGZyb20gdGhlIHBvcnRob2xlcyBhbmQgZGVja3MsIGZhaW50IHNtb2tlIGZyb20gdGhlIGZ1bm5lbHMsIGNvbGQgYXRtb3NwaGVyZSIsICJhdWRpbyI6IFt7InF1aWVuIjogIm5hcnJhZG9yIiwgInRleHRvIjogIkNhdG9yY2UgZGUgYWJyaWwgZGUgbWlsIG5vdmVjaWVudG9zIGRvY2UuIEVzdGEgbm9jaGUsIGVuIGVsIGJhcmNvIG1cdTAwZTFzIGZhbW9zbyBkZWwgbXVuZG8sIGhheSBhbGd1aWVuIHF1ZSBubyBkZWJlclx1MDBlZGEgZXN0YXIgYXF1XHUwMGVkLiJ9XSwgInJvdHVsbyI6IHRydWUsICJhbWJpZW50ZSI6ICJtYXIgZW4gY2FsbWEsIG1vdG9yZXMgbGVqYW5vcyJ9LCB7Im4iOiAyLCAiZHVyYWNpb24iOiAzLjAsICJ0aXBvIjogInZpZGVvIiwgInBlcnNvbmFqZXMiOiBbXSwgInByb21wdCI6ICJOaWdodCBzY2VuZS4gUG9pbnQtb2YtdmlldyBzaG90IGZyb20gaW5zaWRlIGEgZGFyayBuYXJyb3cgc2VydmljZSBjb3JyaWRvciBvZiBhIDE5MTIgb2NlYW4gbGluZXIsIGEgd29vZGVuIGRvb3Igb3BlbmluZyBvbnRvIHRoZSBmaXJzdC1jbGFzcyBwcm9tZW5hZGUgZGVjayBhdCBuaWdodCwgd2FybSBnb2xkZW4gbGlnaHQgZnJvbSB0aGUgZGVjayBsYW1wcyBzcGlsbGluZyBpbiwgYSBmYWludCBjb29sIGN5YW4gZ2xvdyBiZWhpbmQgdGhlIHZpZXdlciwgd29vZGVuIGRlY2sgcGxhbmtzIGFuZCBhIHdoaXRlIHJhaWxpbmcgdmlzaWJsZSB0aHJvdWdoIHRoZSBkb29yLCBzdGFycnkgc2t5IiwgIm1vdmltaWVudG9fdmlkZW8iOiAidGhlIGRvb3Igc2xvd2x5IHN3aW5ncyBvcGVuIGFuZCB0aGUgY2FtZXJhIG1vdmVzIGZvcndhcmQgdGhyb3VnaCBpdCIsICJhdWRpbyI6IFtdLCAiYW1iaWVudGUiOiAicHVlcnRhLCB2aWVudG8ifSwgeyJuIjogMywgImR1cmFjaW9uIjogNC4zLCAidGlwbyI6ICJ2aWRlbyIsICJwZXJzb25hamVzIjogWyJzYW50aSJdLCAidmVydGljYWwiOiB0cnVlLCAicHJvbXB0IjogIk5pZ2h0IHNjZW5lLiBNZWRpdW0gc2hvdCBvZiB7c2FudGl9IHN0ZXBwaW5nIG91dCBvbnRvIHRoZSBmaXJzdC1jbGFzcyBwcm9tZW5hZGUgZGVjayBvZiB0aGUgVGl0YW5pYyBhdCBuaWdodCBpbiAxOTEyLCBsb29raW5nIGFyb3VuZCBjb25mdXNlZCBhbmQgYW1hemVkLCB0aGUgYm93IHRvIHRoZSByaWdodCBvZiB0aGUgZnJhbWUsIHdvb2RlbiBkZWNrLCB3aGl0ZSByYWlsaW5nLCBsaWZlYm9hdHMgaGFuZ2luZyBhYm92ZSwgd2FybSBlbGVjdHJpYyB3YWxsIGxhbXBzLCBicmVhdGggdmlzaWJsZSBpbiB0aGUgY29sZCBhaXIsIHN0YXJyeSBza3kiLCAibW92aW1pZW50b192aWRlbyI6ICJoZSB0YWtlcyBhIHN0ZXAgZm9yd2FyZCBhbmQgdHVybnMgaGlzIGhlYWQgc2xvd2x5LCBsb29raW5nIGFyb3VuZCIsICJhdWRpbyI6IFt7InF1aWVuIjogInNhbnRpIiwgInRpcG8iOiAicGVuc2FtaWVudG8iLCAidGV4dG8iOiAiQXZlIE1hclx1MDBlZGFcdTIwMjYgZXN0byBubyBlcyBlbCBtZXRybyBkZSBNZWRlbGxcdTAwZWRuLiJ9XSwgImFtYmllbnRlIjogInZpZW50bywgbWFyIn0sIHsibiI6IDQsICJkdXJhY2lvbiI6IDIuMCwgInRpcG8iOiAiaW1hZ2VuX2ZpamEiLCAibW92aW1pZW50byI6ICJsZXZlIiwgInBlcnNvbmFqZXMiOiBbInNhbnRpIl0sICJwcm9tcHQiOiAiTmlnaHQgc2NlbmUuIENsb3NlLXVwIGluc2VydCBvZiBhIG1hbidzIGhhbmQgaG9sZGluZyBhIG1vZGVybiBzbWFydHBob25lIHdpdGggYSBibGFjayBjYXNlLCB0aGUgc2NyZWVuIGdsb3dpbmcgY29vbCBibHVlLXdoaXRlIGFuZCBibGFuaywgYWdhaW5zdCB0aGUgZGFyayB3b29kZW4gZGVjayBvZiBhIDE5MTIgb2NlYW4gbGluZXIgYXQgbmlnaHQsIHdhcm0gbGFtcCBsaWdodCBpbiB0aGUgYmFja2dyb3VuZCIsICJzdXBlcnBvc2ljaW9uIjogeyJ0ZXh0byI6ICI0OjA0IiwgInN1YnRleHRvIjogIlNpbiBzZXJ2aWNpbyIsICJ4IjogMC41MTIsICJ5IjogMC40MiwgInRhbWFubyI6IDQ4fSwgImF1ZGlvIjogW10sICJhbWJpZW50ZSI6ICJwaXRpZG8gZGUgZXJyb3Igc3VhdmUifSwgeyJuIjogNSwgImR1cmFjaW9uIjogMy4wLCAidGlwbyI6ICJ2aWRlbyIsICJwZXJzb25hamVzIjogWyJtYXJnYXJldCJdLCAidmVydGljYWwiOiB0cnVlLCAicHJvbXB0IjogIk5pZ2h0IHNjZW5lLiBNZWRpdW0gc2hvdCBvZiB7bWFyZ2FyZXR9IHN0YW5kaW5nIGJ5IHRoZSB3aGl0ZSByYWlsaW5nIG9mIHRoZSBUaXRhbmljJ3MgZmlyc3QtY2xhc3MgcHJvbWVuYWRlIGRlY2sgYXQgbmlnaHQgaW4gMTkxMiwgbG9va2luZyBhdCBzb21lb25lIG9mZi1zY3JlZW4gdG8gdGhlIGxlZnQgd2l0aCBkaXNkYWluIGFuZCBjdXJpb3NpdHksIG9uZSBleWVicm93IHJhaXNlZCwgd2FybSBlbGVjdHJpYyB3YWxsIGxhbXBzLCBzdGFycnkgc2t5LCBjYWxtIGJsYWNrIHNlYSBiZWhpbmQgaGVyLCBicmVhdGggdmlzaWJsZSBpbiB0aGUgY29sZCBhaXIiLCAibW92aW1pZW50b192aWRlbyI6ICJzaGUgc2xvd2x5IGxvb2tzIGhpbSB1cCBhbmQgZG93biIsICJhdWRpbyI6IFtdLCAiYW1iaWVudGUiOiAibVx1MDBmYXNpY2EgZGUgY3VlcmRhIGxlamFuYSJ9LCB7Im4iOiA2LCAiZHVyYWNpb24iOiA1LjYsICJ0aXBvIjogImxpcHN5bmMiLCAicGVyc29uYWplcyI6IFsibWFyZ2FyZXQiXSwgInZlcnRpY2FsIjogdHJ1ZSwgInByb21wdCI6ICJOaWdodCBzY2VuZS4gTWVkaXVtIGNsb3NlLXVwIG9mIHttYXJnYXJldH0gb24gdGhlIFRpdGFuaWMncyBmaXJzdC1jbGFzcyBkZWNrIGF0IG5pZ2h0IGluIDE5MTIsIGZhY2luZyB0aGUgY2FtZXJhIHNsaWdodGx5IHRvIHRoZSBsZWZ0LCBoYXVnaHR5IGV4cHJlc3Npb24sIHdhcm0gbGFtcCBsaWdodCBvbiBoZXIgZmFjZSwgZGFyayBzdGFycnkgYmFja2dyb3VuZCBzb2Z0bHkgYmx1cnJlZCwgZmFjZSBjbGVhcmx5IHZpc2libGUgYW5kIHdlbGwgbGl0IiwgIm1vdmltaWVudG9fdmlkZW8iOiAic2hlIHNwZWFrcyB3aXRoIGEgaGF1Z2h0eSBleHByZXNzaW9uLCBzdWJ0bGUgaGVhZCBtb3ZlbWVudCIsICJhdWRpbyI6IFt7InF1aWVuIjogIm1hcmdhcmV0IiwgInRleHRvIjogIlx1MDBiZlNlIGhhIHBlcmRpZG8sIGNhYmFsbGVybz8gRXN0YSBjdWJpZXJ0YSBlcyBkZSBwcmltZXJhIGNsYXNlLiJ9XSwgImFtYmllbnRlIjogIm1cdTAwZmFzaWNhIGRlIGN1ZXJkYSBsZWphbmEifSwgeyJuIjogNywgImR1cmFjaW9uIjogNi42LCAidGlwbyI6ICJsaXBzeW5jIiwgInBlcnNvbmFqZXMiOiBbInNhbnRpIl0sICJ2ZXJ0aWNhbCI6IHRydWUsICJwcm9tcHQiOiAiTmlnaHQgc2NlbmUuIE1lZGl1bSBjbG9zZS11cCBvZiB7c2FudGl9IG9uIHRoZSBUaXRhbmljJ3MgZmlyc3QtY2xhc3MgZGVjayBhdCBuaWdodCBpbiAxOTEyLCBmYWNpbmcgdGhlIGNhbWVyYSBzbGlnaHRseSB0byB0aGUgcmlnaHQsIG5lcnZvdXMgZnJpZW5kbHkgc21pbGUsIHdhcm0gbGFtcCBsaWdodCBvbiBvbmUgc2lkZSBvZiBoaXMgZmFjZSwgYSBmYWludCBjb29sIGJsdWUgZ2xvdyBmcm9tIGJlbG93IGZyb20gaGlzIHBob25lLCBkYXJrIHN0YXJyeSBiYWNrZ3JvdW5kIHNvZnRseSBibHVycmVkLCBmYWNlIGNsZWFybHkgdmlzaWJsZSBhbmQgd2VsbCBsaXQiLCAibW92aW1pZW50b192aWRlbyI6ICJoZSBzcGVha3Mgd2l0aCBhIG5lcnZvdXMgZnJpZW5kbHkgc21pbGUsIHNtYWxsIGdlc3R1cmVzIiwgImF1ZGlvIjogW3sicXVpZW4iOiAic2FudGkiLCAidGV4dG8iOiAiXHUwMGJmUXVcdTAwZTkgcGFzYSwgcHVlcz8gVHJhbnF1aWxhLCBtaSBzZVx1MDBmMW9yYVx1MjAyNiBPaWdhLCB1bmEgcHJlZ3VudGljYTogXHUwMGJmZXN0ZSBiYXJjbyBjXHUwMGYzbW8gc2UgbGxhbWE/In1dLCAiYW1iaWVudGUiOiAidmllbnRvIn0sIHsibiI6IDgsICJkdXJhY2lvbiI6IDUuMywgInRpcG8iOiAidmlkZW8iLCAicGVyc29uYWplcyI6IFsibWFyZ2FyZXQiXSwgInByb21wdCI6ICJOaWdodCBzY2VuZS4gTWVkaXVtIHNob3QgaW4gcHJvZmlsZSBvZiB7bWFyZ2FyZXR9IGxvb2tpbmcgcHJvdWRseSBvdXQgYXQgdGhlIGNhbG0gYmxhY2sgc2VhIGZyb20gdGhlIFRpdGFuaWMncyByYWlsaW5nIGF0IG5pZ2h0IGluIDE5MTIsIGNoaW4gcmFpc2VkLCB3YXJtIGxhbXAgbGlnaHQgb3V0bGluaW5nIGhlciBwcm9maWxlLCBzdGFycnkgc2t5IiwgIm1vdmltaWVudG9fdmlkZW8iOiAic2hlIGxpZnRzIGhlciBjaGluIHByb3VkbHksIGFsbW9zdCBzdGlsbCIsICJhdWRpbyI6IFt7InF1aWVuIjogIm1hcmdhcmV0IiwgInRleHRvIjogIkVsIFRpdGFuaWMsIHBvciBzdXB1ZXN0by4gRWwgYmFyY28gbVx1MDBlMXMgc2VndXJvIGphbVx1MDBlMXMgY29uc3RydWlkby4ifV0sICJhbWJpZW50ZSI6ICJtYXIifSwgeyJuIjogOSwgImR1cmFjaW9uIjogNC4wLCAidGlwbyI6ICJ2aWRlbyIsICJwZXJzb25hamVzIjogWyJzYW50aSJdLCAidmVydGljYWwiOiB0cnVlLCAicHJvbXB0IjogIk5pZ2h0IHNjZW5lLiBDbG9zZS11cCBvZiB0aGUgZmFjZSBvZiB7c2FudGl9IG9uIHRoZSBUaXRhbmljJ3MgZGVjayBhdCBuaWdodCBpbiAxOTEyLCBoaXMgZXhwcmVzc2lvbiBjb2xsYXBzaW5nIGludG8gc2xvd2x5IGRhd25pbmcgaG9ycm9yLCBleWVzIHdpZGUsIHRoZSBzbWlsZSBkaXNhcHBlYXJpbmcsIHdhcm0gbGFtcCBsaWdodCBmcm9tIHRoZSBzaWRlLCBkYXJrIGJhY2tncm91bmQiLCAibW92aW1pZW50b192aWRlbyI6ICJoaXMgc21pbGUgc2xvd2x5IGZhZGVzIGludG8gaG9ycm9yLCBleWVzIHdpZGVuaW5nIiwgImF1ZGlvIjogW3sicXVpZW4iOiAic2FudGkiLCAidGlwbyI6ICJwZW5zYW1pZW50byIsICJ0ZXh0byI6ICJBeSwgbm8uIE5vLCBubywgbm8sIG5vXHUyMDI2In1dLCAiYW1iaWVudGUiOiAic2lsZW5jaW8iLCAiZmluYWwiOiAibmVncm8ifV19LCAiYXJjaGl2b3MiOiB7InBsYW5vXzAyLmpwZyI6ICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbmVpZG9zdG9yZS9DYW5hbC15b3V0dWJlL2Njci0wMDY0NGU4Yi0xNGI0amkvcGVsaWN1bGFzLWlhL3Byb2R1Y2Npb25lcy9wcnVlYmEtdGVjbmljYS9pbWFnZW5lcy1jb2xhYi12My9wbGFub18wMi5qcGciLCAicGxhbm9fMDMuanBnIjogImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9uZWlkb3N0b3JlL0NhbmFsLXlvdXR1YmUvY2NyLTAwNjQ0ZThiLTE0YjRqaS9wZWxpY3VsYXMtaWEvcHJvZHVjY2lvbmVzL3BydWViYS10ZWNuaWNhL2ltYWdlbmVzLWNvbGFiLXYzL3BsYW5vXzAzLmpwZyIsICJwbGFub18wNS5qcGciOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvaW1hZ2VuZXMtY29sYWItdjMvcGxhbm9fMDUuanBnIiwgInBsYW5vXzA2LmpwZyI6ICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbmVpZG9zdG9yZS9DYW5hbC15b3V0dWJlL2Njci0wMDY0NGU4Yi0xNGI0amkvcGVsaWN1bGFzLWlhL3Byb2R1Y2Npb25lcy9wcnVlYmEtdGVjbmljYS9pbWFnZW5lcy1jb2xhYi12My9wbGFub18wNi5qcGciLCAicGxhbm9fMDcuanBnIjogImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9uZWlkb3N0b3JlL0NhbmFsLXlvdXR1YmUvY2NyLTAwNjQ0ZThiLTE0YjRqaS9wZWxpY3VsYXMtaWEvcHJvZHVjY2lvbmVzL3BydWViYS10ZWNuaWNhL2ltYWdlbmVzLWNvbGFiLXYzL3BsYW5vXzA3LmpwZyIsICJwbGFub18wOC5qcGciOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvaW1hZ2VuZXMtY29sYWItdjMvcGxhbm9fMDguanBnIiwgInBsYW5vXzA5LmpwZyI6ICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vbmVpZG9zdG9yZS9DYW5hbC15b3V0dWJlL2Njci0wMDY0NGU4Yi0xNGI0amkvcGVsaWN1bGFzLWlhL3Byb2R1Y2Npb25lcy9wcnVlYmEtdGVjbmljYS9pbWFnZW5lcy1jb2xhYi12My9wbGFub18wOS5qcGciLCAicGxhbm9fMDFfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDFfMS53YXYiLCAicGxhbm9fMDNfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDNfMS53YXYiLCAicGxhbm9fMDZfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDZfMS53YXYiLCAicGxhbm9fMDdfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDdfMS53YXYiLCAicGxhbm9fMDhfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDhfMS53YXYiLCAicGxhbm9fMDlfMS53YXYiOiAiaHR0cHM6Ly9yYXcuZ2l0aHVidXNlcmNvbnRlbnQuY29tL25laWRvc3RvcmUvQ2FuYWwteW91dHViZS9jY3ItMDA2NDRlOGItMTRiNGppL3BlbGljdWxhcy1pYS9wcm9kdWNjaW9uZXMvcHJ1ZWJhLXRlY25pY2EvYXVkaW8vcGxhbm9fMDlfMS53YXYifSwgImR1cl92b3oiOiB7fX0='))
PLANOS = DATOS["planos"]
import urllib.request
for nombre, url in DATOS["archivos"].items():      # fotogramas y voces se bajan del repositorio
    urllib.request.urlretrieve(url, f"{IN}/{nombre}")
log = lambda *a: print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
sh = lambda c: subprocess.run(c + " > /content/log_comandos.txt 2>&1" if ">" not in c else c, shell=True, check=True)

# ---------------- 0) voces ----------------
VOCES = f"{OUT}/audio"; os.makedirs(VOCES, exist_ok=True)
SCRIPT_VOCES = r"""
import json, os, sys, time, torch, torchaudio, soundfile as sf
PLANOS = json.load(open(sys.argv[1])); OUT = sys.argv[2]; REFS = OUT + "/refs"; os.makedirs(REFS, exist_ok=True)
from kokoro import KPipeline
kp = KPipeline(lang_code="e", repo_id="hexgrad/Kokoro-82M")
SEMILLAS = {"narrador": "em_alex", "santi": "em_santa", "margaret": "ef_dora"}
frase = "Buenas noches. Esta es una muestra de voz para la película, leída con calma y claridad, sin prisa."
for quien, voz in SEMILLAS.items():
    audio = [a for _, _, a in kp(frase, voice=voz)]
    sf.write(f"{REFS}/{quien}.wav", torch.cat([torch.as_tensor(a) for a in audio]).numpy(), 24000)
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
tts = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
AJ = {"narrador": dict(exaggeration=0.35, cfg_weight=0.5), "santi": dict(exaggeration=0.65, cfg_weight=0.4),
      "margaret": dict(exaggeration=0.5, cfg_weight=0.5)}
for pl in PLANOS["planos"]:
    for i, l in enumerate(pl.get("audio", []), 1):
        torch.manual_seed(404 + pl["n"])
        wav = tts.generate(l["texto"], language_id="es", audio_prompt_path=f"{REFS}/{l['quien']}.wav", **AJ[l["quien"]])
        torchaudio.save(f"{OUT}/plano_{pl['n']:02d}_{i}.wav", wav.cpu(), tts.sr)
        print("voz", pl["n"], i, l["quien"], round(wav.shape[-1] / tts.sr, 1), "s", flush=True)
"""
for nombre in list(DATOS["archivos"]):
    if nombre.endswith(".wav"):
        os.replace(f"{IN}/{nombre}", f"{VOCES}/{nombre}")
try:
    if any(f.endswith(".wav") for f in os.listdir(VOCES)):
        raise StopIteration
    log("preparando voces")
    sh("apt-get -qq install -y espeak-ng > /dev/null")
    sh(f"{sys.executable} -m pip install -q uv")
    sh("uv venv -q -p 3.11 /content/vz && uv pip install -q -p /content/vz/bin/python 'chatterbox-tts>=0.1.4' 'kokoro>=0.9' soundfile 'setuptools<81'")
    open("/content/voces.py", "w").write(SCRIPT_VOCES)
    json.dump(PLANOS, open("/content/planos.json", "w"))
    sh(f"/content/vz/bin/python /content/voces.py /content/planos.json {VOCES} > {OUT}/log_voces.txt 2>&1")
    log("voces listas")
except StopIteration:
    log("voces ya incluidas en el cuaderno")
except Exception:
    traceback.print_exc(); open(f"{OUT}/error_voces.txt", "w").write(traceback.format_exc())
def dur_wav(f):
    import wave
    try:
        with wave.open(f) as w: return w.getnframes() / w.getframerate()
    except Exception:
        return 0

# ---------------- 1) vídeo ----------------
sh(f"{sys.executable} -m pip install -q -U 'diffusers>=0.35' 'transformers>=4.51' accelerate ftfy 'imageio[ffmpeg]'")
import torch, html, re, ftfy
from transformers import UMT5EncoderModel, AutoTokenizer
from diffusers import WanImageToVideoPipeline, WanTransformer3DModel, AutoencoderKLWan
from diffusers.utils import export_to_video
from PIL import Image
M = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
RESOLUCIONES = [(896, 512), (768, 448), (640, 352)]   # si falta memoria, se baja
FPS = 24
NEG = ("static, frozen, blurry, distorted face, deformed hands, extra fingers, extra limbs, flicker, morphing, "
       "text, subtitles, watermark, logo, cartoon, low quality")
trabajos = []
for pl in PLANOS["planos"]:
    if pl["tipo"] not in ("video", "lipsync") or not os.path.exists(f"{IN}/plano_{pl['n']:02d}.jpg"):
        continue
    p = f"{pl.get('movimiento_video', 'subtle natural movement')}. {pl['prompt']}"
    for nom, c in PLANOS["personajes"].items():
        p = p.replace("{" + nom + "}", c.get("descripcion", nom))
    trabajos.append((pl, p + ". Cinematic, realistic, natural motion, film grain."))
trabajos.sort(key=lambda x: x[0]["tipo"] != "lipsync")     # primero los que luego llevan labios

def limpiar(t):
    t = ftfy.fix_text(t); t = html.unescape(html.unescape(t)); return re.sub(r"\s+", " ", t).strip()
log("calculando textos con T5")
tok = AutoTokenizer.from_pretrained(M, subfolder="tokenizer")
te = UMT5EncoderModel.from_pretrained(M, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map="cuda")
@torch.no_grad()
def embed(t, L=512):
    x = tok([limpiar(t)], padding="max_length", max_length=L, truncation=True, add_special_tokens=True,
            return_attention_mask=True, return_tensors="pt")
    e = te(x.input_ids.cuda(), x.attention_mask.cuda()).last_hidden_state
    n = int(x.attention_mask.sum()); e[:, n:] = 0
    return e.to(torch.float16).cpu()
neg = embed(NEG); embs = {pl["n"]: embed(p) for pl, p in trabajos}
del te; gc.collect(); torch.cuda.empty_cache()

log("cargando Wan 2.2")
try:
    tr = WanTransformer3DModel.from_pretrained(M, subfolder="transformer", torch_dtype=torch.float16, device_map="cuda")
except Exception:
    tr = WanTransformer3DModel.from_pretrained(M, subfolder="transformer", torch_dtype=torch.float16, low_cpu_mem_usage=True).to("cuda")
vae = AutoencoderKLWan.from_pretrained(M, subfolder="vae", torch_dtype=torch.float32).to("cuda")
try: vae.enable_tiling()
except Exception: pass
try:   # pesos guardados en fp8 y calculados en fp16: la mitad de memoria
    tr.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.float16); log("fp8 activado")
except Exception:
    traceback.print_exc()
pipe = WanImageToVideoPipeline.from_pretrained(M, transformer=tr, vae=vae, text_encoder=None, tokenizer=None,
                                               torch_dtype=torch.float16).to("cuda")
pipe.set_progress_bar_config(disable=True)
metricas = {}
for pl, p in trabajos:
    n = pl["n"]
    if time.time() - T0 > LIMITE:
        log("sin tiempo: me salto el plano", n); continue
    try:
        dur = max(float(pl["duracion"]), dur_wav(f"{VOCES}/plano_{n:02d}_1.wav") + 0.3 if pl["tipo"] == "lipsync" else 0)
        frames = int(min(dur, 5) * FPS) // 4 * 4 + 1
        ti = time.time(); v = None
        for W, H in RESOLUCIONES:
            img = Image.open(f"{IN}/plano_{n:02d}.jpg").convert("RGB")
            s_ = max(W / img.width, H / img.height)
            img = img.resize((round(img.width * s_), round(img.height * s_)), Image.LANCZOS)
            l, t = (img.width - W) // 2, (img.height - H) // 2
            img = img.crop((l, t, l + W, t + H))
            try:
                v = pipe(image=img, prompt_embeds=embs[n].cuda(), negative_prompt_embeds=neg.cuda(), height=H, width=W,
                         num_frames=frames, num_inference_steps=25, guidance_scale=5.0,
                         generator=torch.Generator("cuda").manual_seed(404 + n)).frames[0]
                break
            except torch.OutOfMemoryError:
                log(f"plano {n}: sin memoria a {W}x{H}, pruebo más pequeño"); gc.collect(); torch.cuda.empty_cache()
        if v is None:
            raise RuntimeError("sin memoria en todas las resoluciones")
        export_to_video(v, f"{OUT}/plano_{n:02d}.mp4", fps=FPS)
        brillo = float(sum(f.mean() for f in v[::8]) / len(v[::8]))
        metricas[n] = {"segundos": round(time.time() - ti), "fotogramas": frames, "tam": f"{W}x{H}", "brillo_medio": round(brillo, 3)}
        log(f"plano {n} listo", metricas[n])
    except Exception:
        traceback.print_exc(); metricas[n] = {"error": traceback.format_exc()[-800:]}; gc.collect(); torch.cuda.empty_cache()
json.dump(metricas, open(f"{OUT}/metricas_video.json", "w"), indent=1)
del pipe, tr, vae; gc.collect(); torch.cuda.empty_cache()

# ---------------- 2) labios ----------------
try:
    lip = [pl for pl in PLANOS["planos"] if pl["tipo"] == "lipsync"
           and os.path.exists(f"{OUT}/plano_{pl['n']:02d}.mp4") and os.path.exists(f"{VOCES}/plano_{pl['n']:02d}_1.wav")]
    if lip and time.time() - T0 < LIMITE + 1800:
        log("preparando LatentSync 1.5")
        LS = "/content/LatentSync"
        if not os.path.exists(LS):
            sh(f"git clone -q https://github.com/bytedance/LatentSync.git {LS}")
        sh(f"{sys.executable} -m pip install -q uv")
        sh(f"cd {LS} && uv venv -q -p 3.10 /content/lsenv && uv pip install -q -p /content/lsenv/bin/python "
           f"--index-strategy unsafe-best-match -r requirements.txt")
        sh(f"cd {LS} && /content/lsenv/bin/python -c \"from huggingface_hub import hf_hub_download as d; "
           f"[d('ByteDance/LatentSync-1.5', f, local_dir='checkpoints') for f in ('latentsync_unet.pt', 'whisper/tiny.pt')]\"")
        for pl in lip:
            n = pl["n"]; ti = time.time()
            try:
                sh(f"cd {LS} && /content/lsenv/bin/python -m scripts.inference --unet_config_path configs/unet/stage2.yaml "
                   f"--inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps 20 --guidance_scale 1.5 "
                   f"--video_path {OUT}/plano_{n:02d}.mp4 --audio_path {VOCES}/plano_{n:02d}_1.wav "
                   f"--video_out_path {OUT}/plano_{n:02d}_labios.mp4 > {OUT}/log_labios_{n:02d}.txt 2>&1")
                log(f"labios plano {n} · {time.time()-ti:.0f} s")
            except Exception:
                traceback.print_exc()
except Exception:
    traceback.print_exc(); open(f"{OUT}/error_labios.txt", "w").write(traceback.format_exc())
log("FIN")


In [ ]:
import shutil
shutil.make_archive('/content/neido-video', 'zip', '/content/entrega')
from google.colab import files
files.download('/content/neido-video.zip')
print('LISTO: se descarga neido-video.zip')
